Compare Zero-Shot NLI vs GoEmotions
=======================================
Both classifiers run in `04_classification.ipynb` (§ 4A and § 4B) over the same
songs from `03_lyrics_trans.csv`, under a **shared scoring contract** — so what
differs between them is the model and the taxonomy, not the plumbing:

| | zero-shot (`04.1_*.csv`) | GoEmotions (`04.2_*.csv`) |
|---|---|---|
| scoring | independent per-label prob. in [0, 1] (`ZEROSHOT_MULTI_LABEL=True`) | independent per-label prob. in [0, 1] (sigmoid head) |
| `unclassified` | no scoreable lyrics | no scoreable lyrics |
| confidence | `low_confidence` flag at 0.30, never dropped | same |
| **labels** | **10, hand-picked for songs** | **28, fixed GoEmotions** |
| **model** | **`bart-large-mnli`, zero-shot NLI** | **`roberta-base-go_emotions`, supervised** |
| **`neutral` label** | **absent** | **present** |

### What is and isn't comparable

Even with the contract in place, three things stay in the way:

1. **Calibration.** RoBERTa's sigmoids sit far below bart-mnli's entailment
   probabilities (median top score ~0.50 vs ~0.97). Raw magnitudes are the same
   *kind* of quantity now, but not the same *scale*.
2. **Taxonomy overlap.** Only four label names appear in both sets.
3. **`neutral`.** No zero-shot equivalent exists, so GoEmotions' dominant label
   is taken as `dominant_emotion_emotive` (best non-neutral) wherever the two
   distributions are put side by side.

So this notebook leans on **rank and z-score comparisons**, which are invariant
to (1), and treats absolute-score comparisons as secondary. The headline test is
§ 5: do the two classifiers draw the *same regional map*, regardless of scale?

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"

zeroshot   = pd.read_csv(PROCESSED / "04.1_emotion_scores_zeroshot.csv")
goemotions = pd.read_csv(PROCESSED / "04.2_emotion_scores_goemotions.csv")
titles     = pd.read_csv(PROCESSED / "00_titles.csv")[["spotify_uri", "region"]]

# Label names present in BOTH taxonomies.
SHARED_LABELS = ["love", "joy", "grief", "anger"]

# Near-equivalents: a judgment call, NOT a model output. The zero-shot vocabulary was
# built for songs and splits hairs GoEmotions doesn't, so these pairs are
# "closest available", not synonyms. Reported separately from SHARED_LABELS
# throughout so the two never get conflated.
NEAR_EQUIVALENT = {
    "heartbreak": "sadness",
    "sensual":    "desire",
    "hope":       "optimism",
    "despair":    "disappointment",
}

print(f"zero-shot:  {len(zeroshot)} songs, "
      f"{sum(c.startswith('emotion_') for c in zeroshot.columns)} labels")
print(f"goemotions: {len(goemotions)} songs, "
      f"{sum(c.startswith('emotion_') for c in goemotions.columns)} labels")

zero-shot:  1105 songs, 10 labels
goemotions: 1105 songs, 28 labels


### 1. Contract check

Before comparing anything, verify the two files actually satisfy the shared
contract. If any of these fail, the rest of the notebook is not meaningful —
re-run `04_classification.ipynb` rather than reading on.

In [2]:
zs_cols = [c for c in zeroshot.columns if c.startswith("emotion_")]
ge_cols = [c for c in goemotions.columns if c.startswith("emotion_")]

checks = {}

checks["identical track sets"] = set(zeroshot.spotify_uri) == set(goemotions.spotify_uri)

# Independent scoring: a softmax over labels would force every row to sum to 1.
def is_label_softmax(df, cols):
    """True if every scored row sums to ~1 — i.e. scores compete across labels."""
    scored_rows = df.loc[df[cols].max(axis=1) > 0, cols]
    return bool(np.isclose(scored_rows.sum(axis=1), 1.0, atol=0.02).all())

# Contract point 1 is ADVISORY, not fatal. If 04's ZEROSHOT_MULTI_LABEL is set
# to False the zero-shot scores become a label softmax — magnitudes then aren't
# comparable to GoEmotions' sigmoids, but ranks still are, so this notebook
# degrades to rank-based comparisons instead of refusing to run.
INDEPENDENT_SCORES = not (is_label_softmax(zeroshot, zs_cols)
                          or is_label_softmax(goemotions, ge_cols))

checks["both carry dominant_score / low_confidence"] = all(
    c in d.columns for d in (zeroshot, goemotions)
    for c in ("dominant_score", "low_confidence")
)

# 'unclassified' must mean "no scoreable lyrics" in BOTH — i.e. exactly the
# all-zero rows, with no confidence-based dropping mixed in.
zs_unc = set(zeroshot.loc[zeroshot.dominant_emotion == "unclassified", "spotify_uri"])
ge_unc = set(goemotions.loc[goemotions.dominant_emotion == "unclassified", "spotify_uri"])
checks["zero-shot: unclassified == all-zero rows"] = zs_unc == set(
    zeroshot.loc[zeroshot[zs_cols].max(axis=1) <= 0, "spotify_uri"])
checks["goemotions: unclassified == all-zero rows"] = ge_unc == set(
    goemotions.loc[goemotions[ge_cols].max(axis=1) <= 0, "spotify_uri"])

print("Fatal checks (comparison is invalid without these):")
for name, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {name}")

print(f"\nunclassified — zero-shot {len(zs_unc)}, goemotions {len(ge_unc)}, "
      f"shared {len(zs_unc & ge_unc)}, only-one-fork {len(zs_unc ^ ge_unc)}")

if not all(checks.values()):
    raise AssertionError(
        "Shared scoring contract violated — re-run 04_classification.ipynb before "
        "comparing. See its header for what the contract requires."
    )

print(f"\nIndependent per-label scoring (contract point 1): "
      f"{'YES' if INDEPENDENT_SCORES else 'NO'}")
if not INDEPENDENT_SCORES:
    print("  → One fork uses a label softmax (ZEROSHOT_MULTI_LABEL = False in 04).")
    print("    Magnitude comparisons in section 4 are NOT meaningful; read")
    print("    spearman_rho only, and rely on section 5, which is rank-based.")

Fatal checks (comparison is invalid without these):
  PASS  identical track sets
  PASS  both carry dominant_score / low_confidence
  PASS  zero-shot: unclassified == all-zero rows
  PASS  goemotions: unclassified == all-zero rows

unclassified — zero-shot 107, goemotions 107, shared 107, only-one-fork 0

Independent per-label scoring (contract point 1): YES


### 2. Coverage and confidence

With `unclassified` meaning the same thing in both, coverage is near-identical
by construction — which is the point. The interesting number is no longer *how
many* songs each fork kept, but **how confident each model is on the songs they
both scored**. That is a genuine model property rather than a threshold choice.

In [3]:
# Rename every emotion column explicitly before merging. Relying on pandas'
# `suffixes=` would only disambiguate the OVERLAPPING names (love/joy/grief/
# anger), silently leaving fork-unique labels like `sadness` or `heartbreak`
# unsuffixed — and any near-equivalent lookup below would then miss.
KEEP = ["spotify_uri", "dominant_emotion", "dominant_score", "low_confidence"]

zs_r = zeroshot.rename(columns={c: f"{c}_zs" for c in zs_cols})[
    KEEP + [f"{c}_zs" for c in zs_cols]]
ge_r = goemotions.rename(columns={c: f"{c}_ge" for c in ge_cols})[
    KEEP + [f"{c}_ge" for c in ge_cols]]

merged = zs_r.merge(ge_r, on="spotify_uri", suffixes=("_zs", "_ge"), validate="1:1")
scored = merged[
    (merged.dominant_emotion_zs != "unclassified")
    & (merged.dominant_emotion_ge != "unclassified")
].copy()

coverage = pd.DataFrame({
    "zero-shot (04.1)": [
        len(zeroshot),
        (zeroshot.dominant_emotion == "unclassified").sum(),
        zeroshot.low_confidence.sum(),
        zeroshot.dominant_score.median(),
    ],
    "goemotions (04.2)": [
        len(goemotions),
        (goemotions.dominant_emotion == "unclassified").sum(),
        goemotions.low_confidence.sum(),
        goemotions.dominant_score.median(),
    ],
}, index=["songs", "unclassified (no lyrics)", "low_confidence (< 0.30)",
          "median dominant_score"]).round(3)

display(coverage)
print(f"Songs scored by both: {len(scored)}")

,zero-shot (04.1),goemotions (04.2)
songs,1105.000,1105.000
unclassified (no lyrics),107.000,107.000
low_confidence (< 0.30),5.000,94.000
median dominant_score,0.971,0.536


Songs scored by both: 998


GoEmotions carries far more `low_confidence` rows at the same 0.30 bar. Read
that as calibration, not incompetence: a supervised sigmoid head trained on
short Reddit comments produces conservative probabilities on 300-word lyrics,
while zero-shot NLI entailment scores run hot almost everywhere. It is a reason
to prefer rank-based comparisons below — not evidence that one model is right.

### 3. Dominant emotion distribution

Each fork in its own taxonomy. GoEmotions uses `dominant_emotion_emotive`
(best non-neutral) so the two are at least both answering "which *emotion*",
with the neutral share reported separately.

In [4]:
# Best non-neutral GoEmotions label, keyed by spotify_uri so it can be joined
# onto `merged` by key rather than by positional index.
ge_emotive_cols = [c for c in ge_cols if c != "emotion_neutral"]
ge_emotive_by_uri = (
    goemotions[ge_emotive_cols].idxmax(axis=1).str.replace("emotion_", "", regex=False)
      .where(goemotions.dominant_emotion != "unclassified")
      .set_axis(goemotions.spotify_uri)
)
ge_emotive = ge_emotive_by_uri.reset_index(drop=True)

print("=== Zero-shot (04.1) — 10 labels ===")
display(zeroshot.dominant_emotion.value_counts())

print("=== GoEmotions (04.2) — best non-neutral of 27 ===")
display(ge_emotive.value_counts().head(15))

neutral_share = (goemotions.dominant_emotion == "neutral").mean()
print(f"GoEmotions called {neutral_share:.1%} of songs 'neutral' outright "
      f"(no zero-shot counterpart exists).")

=== Zero-shot (04.1) — 10 labels ===


dominant_emotion
longing         348
sensual         207
love            131
lonely          112
heartbreak      109
unclassified    107
anger            57
despair          10
hope              9
joy               8
grief             7
Name: count, dtype: int64

=== GoEmotions (04.2) — best non-neutral of 27 ===


love              279
sadness           116
approval           96
annoyance          68
desire             67
amusement          65
curiosity          61
disappointment     40
admiration         36
disapproval        25
confusion          23
joy                21
caring             20
excitement         15
fear               14
Name: count, dtype: int64

GoEmotions called 31.2% of songs 'neutral' outright (no zero-shot counterpart exists).


### 4. Score agreement on shared labels

For the four label names in both taxonomies. Pearson answers "do the magnitudes
track", Spearman "do the *rankings* track". Spearman is the trustworthy one
here — it is unaffected by the calibration gap, whereas Pearson is dragged down
by it even when the models fully agree about ordering.

In [5]:
rows = []
for label, kind in ([(l, "shared") for l in SHARED_LABELS]
                    + [(l, "near-equiv") for l in NEAR_EQUIVALENT]):
    ge_label = label if kind == "shared" else NEAR_EQUIVALENT[label]
    a = scored[f"emotion_{label}_zs"]
    b = scored[f"emotion_{ge_label}_ge"]
    rows.append({
        "zero-shot label": label,
        "goemotions label": ge_label,
        "pair": kind,
        "pearson_r": pearsonr(a, b)[0],
        "spearman_rho": spearmanr(a, b)[0],
        "mean score (zs)": a.mean(),
        "mean score (ge)": b.mean(),
    })

label_agreement = pd.DataFrame(rows).round(3)
if not INDEPENDENT_SCORES:
    print("Scores are not on a common footing — dropping magnitude columns; "
          "read spearman_rho only.")
    label_agreement = label_agreement.drop(
        columns=["pearson_r", "mean score (zs)", "mean score (ge)"]
    )
display(label_agreement)

,zero-shot label,goemotions label,pair,pearson_r,spearman_rho,mean score (zs),mean score (ge)
0,love,love,shared,0.480,0.543,0.565,0.218
1,joy,joy,shared,0.300,0.336,0.165,0.033
2,grief,grief,shared,0.434,0.488,0.421,0.004
3,anger,anger,shared,0.337,0.351,0.346,0.022
4,heartbreak,sadness,near-equiv,0.459,0.579,0.502,0.103
5,sensual,desire,near-equiv,-0.070,-0.044,0.669,0.076
6,hope,optimism,near-equiv,0.228,0.228,0.258,0.029
7,despair,disappointment,near-equiv,0.473,0.498,0.328,0.072


The `mean score` columns show the calibration gap directly — same label,
same songs, systematically different magnitude. Compare `spearman_rho` across
rows, and ignore the absolute means except as evidence of why you should.

`near-equiv` rows depend on a mapping I chose, not on anything either model
asserted. Treat them as a hypothesis about the taxonomies, not a measurement.

### 5. Do the two classifiers draw the same regional map? *(the headline test)*

This is the comparison the calibration gap cannot distort. Within each fork,
z-score every shared label across regions — turning "Japan scores 0.83 on love"
into "Japan is 1.2 SD above this classifier's own regional average for love".
Then ask whether the two forks' regional profiles agree.

If they do, the classifiers are telling the same story about regional emotional
character in different vocabularies, and the taxonomy choice is largely
cosmetic for regional analysis. If they don't, taxonomy choice is a real
confound and the downstream conclusion depends on which classifier you picked —
which is the thing worth knowing before publishing any regional claim.

In [6]:
def regional_z(df, cols, labels):
    """Mean score per region, z-scored per label ACROSS regions (within-fork)."""
    m = titles.merge(df[["spotify_uri"] + cols], on="spotify_uri", how="inner")
    means = m.groupby("region")[cols].mean()
    z = (means - means.mean()) / means.std(ddof=0)
    z.columns = labels
    return z

zs_z = regional_z(zeroshot, [f"emotion_{l}" for l in SHARED_LABELS], SHARED_LABELS)
ge_z = regional_z(goemotions, [f"emotion_{l}" for l in SHARED_LABELS], SHARED_LABELS)

per_label = pd.DataFrame({
    "pearson_r": [pearsonr(zs_z[l], ge_z[l])[0] for l in SHARED_LABELS],
    "spearman_rho": [spearmanr(zs_z[l], ge_z[l])[0] for l in SHARED_LABELS],
    "max |z| gap": [(zs_z[l] - ge_z[l]).abs().max() for l in SHARED_LABELS],
}, index=SHARED_LABELS).round(3)

print("Agreement on the regional profile, per shared label:")
display(per_label)

flat = pearsonr(zs_z.values.ravel(), ge_z.values.ravel())[0]
print(f"\nOverall agreement across all region x label cells: r = {flat:.3f}")
print("  r > 0.7  → same regional map, different vocabulary")
print("  r < 0.3  → taxonomy choice materially changes the regional conclusion")

print("\nSide-by-side regional z-scores (zs | ge):")
display(zs_z.join(ge_z, lsuffix="_zs", rsuffix="_ge")[
    [c for l in SHARED_LABELS for c in (f"{l}_zs", f"{l}_ge")]
].round(2))

Agreement on the regional profile, per shared label:


,pearson_r,spearman_rho,max |z| gap
love,0.672,0.286,1.434
joy,-0.027,-0.071,3.003
grief,-0.402,-0.286,3.820
anger,0.371,0.333,1.665



Overall agreement across all region x label cells: r = 0.153
  r > 0.7  → same regional map, different vocabulary
  r < 0.3  → taxonomy choice materially changes the regional conclusion

Side-by-side regional z-scores (zs | ge):


,love_zs,love_ge,joy_zs,joy_ge,grief_zs,grief_ge,anger_zs,anger_ge
region,,,,,,,,
Argentina,-1.05,-0.06,-1.59,-0.60,0.22,-0.15,0.48,-0.40
Colombia,0.03,1.16,-0.45,0.86,-0.75,0.38,0.95,-0.18
Global,0.98,0.31,0.27,0.05,1.05,0.12,-0.02,0.59
Japan,0.78,1.37,0.73,2.15,-1.52,2.31,-2.16,-0.71
Singapore,1.13,-0.30,0.87,-0.59,0.53,-0.24,-0.51,-0.01
Spain,0.24,-0.08,-0.83,-0.19,-0.10,-0.48,1.18,-0.06
Taiwan,-1.99,-2.12,1.67,-1.33,-1.05,-1.38,-0.46,-1.44
USA,-0.14,-0.28,-0.66,-0.35,1.62,-0.56,0.54,2.21


### 6. Dominant-emotion agreement

Of the songs GoEmotions assigns to one of the four shared labels, how often does
zero-shot pick the same one? Expect this to stay low, and note that it is the
*least* informative comparison here — 04.1's `sensual` / `longing` / `heartbreak`
have no GoEmotions equivalent and absorb most of its dominant picks, so
disagreement mostly measures taxonomy resolution, not disagreement about
content. Kept for completeness; § 5 is the one to act on.

In [7]:
subset = scored.assign(ge_emotive=ge_emotive.reindex(scored.index))
subset = subset[subset.ge_emotive.isin(SHARED_LABELS)]

exact = (subset.dominant_emotion_zs == subset.ge_emotive).mean()
print(f"Songs where GoEmotions' emotive dominant is a shared label: {len(subset)}")
print(f"Exact agreement with zero-shot's dominant: {exact:.1%}")

print("\nWhat zero-shot picked instead:")
display(
    pd.crosstab(subset.ge_emotive, subset.dominant_emotion_zs)
)

Songs where GoEmotions' emotive dominant is a shared label: 305
Exact agreement with zero-shot's dominant: 29.8%

What zero-shot picked instead:


dominant_emotion_zs,anger,despair,heartbreak,hope,joy,lonely,longing,love,sensual
ge_emotive,,,,,,,,,
anger,3,1,0,0,0,0,0,0,1
joy,0,0,2,0,3,0,9,2,5
love,8,1,20,1,1,12,97,85,54


### 7. Regional summary export

In [8]:
shared_cols = [f"emotion_{l}" for l in SHARED_LABELS]

zs_raw = titles.merge(zeroshot[["spotify_uri"] + shared_cols], on="spotify_uri").groupby("region").mean(numeric_only=True)
ge_raw = titles.merge(goemotions[["spotify_uri"] + shared_cols], on="spotify_uri").groupby("region").mean(numeric_only=True)
zs_raw.columns = [f"{c.replace('emotion_', '')}_zeroshot_raw" for c in zs_raw.columns]
ge_raw.columns = [f"{c.replace('emotion_', '')}_goemotions_raw" for c in ge_raw.columns]

out = (
    zs_raw.join(ge_raw)
          .join(zs_z.add_suffix("_zeroshot_z"))
          .join(ge_z.add_suffix("_goemotions_z"))
)
out = out[[c for l in SHARED_LABELS for c in (
    f"{l}_zeroshot_raw", f"{l}_goemotions_raw",
    f"{l}_zeroshot_z", f"{l}_goemotions_z")]].round(3)

display(out)
output_path = PROCESSED / "07_classifier_comparison_regional.csv"
out.to_csv(output_path)
print(f"Saved to {output_path}")
print("\nUse the _z columns for cross-classifier reads; the _raw columns are "
      "fork-local and differ in scale by construction.")

,love_zeroshot_raw,love_goemotions_raw,love_zeroshot_z,love_goemotions_z,joy_zeroshot_raw,joy_goemotions_raw,joy_zeroshot_z,joy_goemotions_z,grief_zeroshot_raw,grief_goemotions_raw,grief_zeroshot_z,grief_goemotions_z,anger_zeroshot_raw,anger_goemotions_raw,anger_zeroshot_z,anger_goemotions_z
region,,,,,,,,,,,,,,,,
Argentina,0.482,0.195,-1.048,-0.057,0.108,0.024,-1.588,-0.598,0.405,0.003,0.216,-0.150,0.349,0.018,0.481,-0.396
Colombia,0.534,0.261,0.026,1.160,0.143,0.039,-0.452,0.857,0.350,0.003,-0.747,0.383,0.383,0.019,0.953,-0.184
Global,0.581,0.215,0.985,0.314,0.165,0.030,0.266,0.050,0.452,0.003,1.048,0.119,0.312,0.025,-0.024,0.589
Japan,0.571,0.272,0.784,1.366,0.180,0.052,0.729,2.150,0.307,0.005,-1.515,2.305,0.157,0.016,-2.165,-0.713
Singapore,0.588,0.182,1.135,-0.299,0.184,0.024,0.872,-0.590,0.423,0.003,0.529,-0.245,0.277,0.021,-0.513,-0.011
Spain,0.545,0.194,0.243,-0.076,0.132,0.028,-0.831,-0.186,0.387,0.003,-0.101,-0.475,0.400,0.020,1.185,-0.056
Taiwan,0.437,0.083,-1.988,-2.124,0.208,0.017,1.668,-1.335,0.333,0.002,-1.047,-1.382,0.281,0.011,-0.460,-1.439
USA,0.526,0.182,-0.138,-0.285,0.137,0.026,-0.663,-0.347,0.485,0.003,1.617,-0.556,0.353,0.036,0.544,2.209


Saved to /Users/wednesday/Documents/GitHub/genius/data/processed/07_classifier_comparison_regional.csv

Use the _z columns for cross-classifier reads; the _raw columns are fork-local and differ in scale by construction.


### Summary

- **Coverage is no longer a differentiator.** It used to look like zero-shot
  classified ~93 more songs than GoEmotions; that was entirely the `> 0` vs
  `>= 0.30` threshold mismatch. Under the shared contract both drop the same
  ~107 empty-lyric songs.
- **Confidence is** a differentiator: GoEmotions flags far more songs as
  low-confidence at the same bar. This is calibration (a supervised sigmoid
  trained on short Reddit comments, applied to long lyrics), not accuracy, and
  it is the reason to prefer rank/z-score comparisons over raw magnitudes.
- **§ 5 is the finding that matters.** If the regional z-score profiles agree,
  regional conclusions are robust to the taxonomy choice and you can pick a
  classifier on other grounds (speed, label interpretability). If they don't,
  every regional claim needs the classifier named alongside it.
- **Dominant-emotion disagreement is expected and largely uninformative** —
  04.1 was built with a song-specific vocabulary that GoEmotions collapses into
  `love` / `sadness`. That's a resolution difference, not a contradiction.
- **Neither classifier is "more correct".** 04.1 gives interpretable
  song-emotion labels; 04.2 gives a supervised, reproducible, fixed taxonomy
  with published benchmarks. Which is better depends on whether the downstream
  work needs song-specific nuance or defensible provenance.